In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# import scienceplots

# plt.style.use(["science", "muted", "ieee", "no-latex"])
ICML_COLORS = ("#1f77b4", "#ff7f0e", "#2ca02c", "#d62728")


def apply_icml_style():
    """Applies ICML-style Matplotlib defaults for compact figures."""
    plt.rcParams.update({
        "figure.figsize": (4.5, 3.5),
        "figure.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.top": False,
        "ytick.right": False,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.0,
        "lines.markersize": 4,
        "grid.alpha": 0.2,
        "mathtext.fontset": "stix",
    })

# -----------------------
# Data (Twitter15/16 @ 30 mins): mean ± 95% CI
# -----------------------
T_vals = np.array([100, 200, 400])
T_stride = np.array([10, 20, 40])
T_mean_15 = np.array([0.803, 0.812, 0.813])
T_ci_15 = np.array([0.005, 0.005, 0.005])
T_mean_16 = np.array([0.836, 0.845, 0.846])
T_ci_16 = np.array([0.004, 0.003, 0.003])

s_vals = np.array([1.0, 2.0, 3.0, 5.0])
s_mean_15 = np.array([0.803, 0.809, 0.812, 0.809])
s_ci_15 = np.array([0.005, 0.005, 0.005, 0.005])
s_mean_16 = np.array([0.836, 0.842, 0.845, 0.842])
s_ci_16 = np.array([0.004, 0.003, 0.003, 0.004])

lam_vals = np.array([0.0, 0.2, 0.5, 1.0])
lam_mean_15 = np.array([0.801, 0.807, 0.812, 0.808])
lam_ci_15 = np.array([0.005, 0.005, 0.005, 0.005])
lam_mean_16 = np.array([0.833, 0.840, 0.845, 0.841])
lam_ci_16 = np.array([0.004, 0.003, 0.003, 0.004])

apply_icml_style()

# -----------------------
# Helpers
# -----------------------


def save_line_with_ci(x, y, ci, xlabel, title, out_path, tick_formatter=None):
    """Saves a line plot with confidence intervals.

    Args:
        x: Iterable of x-axis values.
        y: Iterable of y-axis means.
        ci: Iterable of 95% confidence interval widths.
        xlabel: Label for the x-axis.
        title: Plot title text.
        out_path: Destination file path for the saved figure.
        tick_formatter: Callable to format tick labels.
    """
    formatter = tick_formatter or (lambda values: [str(v) for v in values])
    fig, ax = plt.subplots()

    ax.errorbar(x, y, yerr=ci, fmt="o-", capsize=2.5, color=ICML_COLORS[0])
    ax.axhline(baseline, linestyle="--", linewidth=1, color="#6e6e6e")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Macro-F1")
    # ax.set_title(title)

    y_min = float(np.min(y - ci))
    y_max = float(np.max(y + ci))
    pad = max(0.001, 0.15 * (y_max - y_min))
    ax.set_ylim(y_min - pad, y_max + pad)

    formatted_ticks = formatter(x)
    ax.set_xticks(x)
    ax.set_xticklabels(formatted_ticks)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", format="pdf")
    plt.close(fig)


def save_grouped_bar_with_ci(x, y1, ci1, y2, ci2, xlabel, title, out_path, 
                              tick_formatter=None, label1="Twitter15", label2="Twitter16",
                              color1=None, color2=None):
    """Saves a grouped bar chart with confidence intervals for two datasets.

    Args:
        x: Iterable of category values.
        y1: Iterable of y-axis means for first dataset.
        ci1: Iterable of 95% confidence interval widths for first dataset.
        y2: Iterable of y-axis means for second dataset.
        ci2: Iterable of 95% confidence interval widths for second dataset.
        xlabel: Label for the x-axis.
        title: Plot title text.
        out_path: Destination file path for the saved figure.
        tick_formatter: Callable to format tick labels.
        label1: Label for the first dataset.
        label2: Label for the second dataset.
        color1: Color for the first dataset bars.
        color2: Color for the second dataset bars.
    """
    formatter = tick_formatter or (lambda values: [str(v) for v in values])
    bar_color1 = color1 or ICML_COLORS[0]
    bar_color2 = color2 or ICML_COLORS[1]
    fig, ax = plt.subplots()

    idx = np.arange(len(x))
    width = 0.35
    
    bars1 = ax.bar(idx - width/2, y1, width, yerr=ci1, capsize=2.5, 
                   color=bar_color1, label=label1)
    bars2 = ax.bar(idx + width/2, y2, width, yerr=ci2, capsize=2.5, 
                   color=bar_color2, label=label2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Macro-F1")
    # ax.set_title(title)
    ax.legend(loc="best")
    ax.tick_params(top=False, right=False)

    ax.set_xticks(idx)
    ax.set_xticklabels(formatter(x))

    y_min = float(np.min([np.min(y1 - ci1), np.min(y2 - ci2)]))
    y_max = float(np.max([np.max(y1 + ci1), np.max(y2 + ci2)]))
    pad = max(0.001, 0.15 * (y_max - y_min))
    ax.set_ylim(y_min - pad, y_max + pad)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", format="pdf")
    plt.close(fig)


# -----------------------
# Generate grouped bar charts for each hyperparameter sweep
# -----------------------
T_tick_formatter = lambda values: [fr"{t}\n(stride {st})" for t, st in zip(values, T_stride)]

save_grouped_bar_with_ci(
    x=T_vals,
    y1=T_mean_15,
    ci1=T_ci_15,
    y2=T_mean_16,
    ci2=T_ci_16,
    xlabel=r"Diffusion horizon $T$",
    title=r"Horizon sweep (matched stride)",
    out_path="Output/figures/hyper_sweep_T.pdf",
    tick_formatter=T_tick_formatter,
    color1=ICML_COLORS[0],
    color2=ICML_COLORS[1],
)

save_grouped_bar_with_ci(
    x=s_vals,
    y1=s_mean_15,
    ci1=s_ci_15,
    y2=s_mean_16,
    ci2=s_ci_16,
    xlabel=r"CFG guidance scale $s$",
    title=r"Guidance sweep",
    out_path="Output/figures/hyper_sweep_s.pdf",
    color1=ICML_COLORS[0],
    color2=ICML_COLORS[1],
)

save_grouped_bar_with_ci(
    x=lam_vals,
    y1=lam_mean_15,
    ci1=lam_ci_15,
    y2=lam_mean_16,
    ci2=lam_ci_16,
    xlabel=r"Diffusion loss weight $\lambda_{\mathrm{diff}}$",
    title=r"Loss-weight sweep",
    out_path="Output/figures/hyper_sweep_lambda.pdf",
    color1=ICML_COLORS[0],
    color2=ICML_COLORS[1],
)

print("Saved: hyper_sweep_T.pdf, hyper_sweep_s.pdf, hyper_sweep_lambda.pdf")


Saved: hyper_sweep_T.pdf, hyper_sweep_s.pdf, hyper_sweep_lambda.pdf


In [21]:
import matplotlib.pyplot as plt
import numpy as np

ICML_COLORS = ("#1f77b4", "#ff7f0e", "#2ca02c", "#d62728")


def apply_icml_style():
    """Applies ICML-style Matplotlib defaults for compact figures."""
    plt.rcParams.update({
        "figure.figsize": (4.5, 3.5),
        "figure.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.top": False,
        "ytick.right": False,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.0,
        "lines.markersize": 4,
        "grid.alpha": 0.2,
        "mathtext.fontset": "stix",
    })


# -----------------------
# Data: Transfer evaluation (macro-F1, mean ± 95% CI)
# Reordered: D$^2$, DGA-Fake, and RumorDiffusion grouped together
# -----------------------
methods = ["BiGCN", "TGAT", "CG-VAE+GCN", "CTDM-syn", "D$^2$", "DGA-Fake", "RumorDiffusion"]

# Original order indices: [0, 1, 2, 3, 4, 5, 6] -> [BiGCN, TGAT, D$^2$, CG-VAE+GCN, CTDM-syn, DGA-Fake, RumorDiffusion]
# New order indices:      [0, 1, 2, 3, 4, 5, 6] -> [BiGCN, TGAT, CG-VAE+GCN, CTDM-syn, D$^2$, DGA-Fake, RumorDiffusion]
# Mapping: 0->0, 1->1, 3->2, 4->3, 2->4, 5->5, 6->6
old_to_new = [0, 1, 3, 4, 2, 5, 6]

# T15→T16
t15_t16_mean_old = np.array([0.760, 0.774, 0.784, 0.788, 0.795, 0.790, 0.808])
t15_t16_ci_old = np.array([0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.005])
t15_t16_mean = t15_t16_mean_old[old_to_new]
t15_t16_ci = t15_t16_ci_old[old_to_new]

# T16→T15
t16_t15_mean_old = np.array([0.772, 0.785, 0.794, 0.798, 0.806, 0.801, 0.817])
t16_t15_ci_old = np.array([0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.005])
t16_t15_mean = t16_t15_mean_old[old_to_new]
t16_t15_ci = t16_t15_ci_old[old_to_new]

# Weibo→PHEME
weibo_pheme_mean_old = np.array([0.842, 0.852, 0.854, 0.856, 0.858, 0.863, 0.872])
weibo_pheme_ci_old = np.array([0.007, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006])
weibo_pheme_mean = weibo_pheme_mean_old[old_to_new]
weibo_pheme_ci = weibo_pheme_ci_old[old_to_new]

# PHEME→Weibo
pheme_weibo_mean_old = np.array([0.902, 0.910, 0.910, 0.912, 0.914, 0.918, 0.925])
pheme_weibo_ci_old = np.array([0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.004])
pheme_weibo_mean = pheme_weibo_mean_old[old_to_new]
pheme_weibo_ci = pheme_weibo_ci_old[old_to_new]


apply_icml_style()


def save_transfer_bar_with_ci(methods, means, cis, xlabel, title, out_path, 
                               highlight_idx=None, colors=None):
    """Saves a bar chart with confidence intervals for transfer evaluation.

    Args:
        methods: List of method names.
        means: Array of y-axis means.
        cis: Array of 95% confidence interval widths.
        xlabel: Label for the x-axis.
        title: Plot title text.
        out_path: Destination file path for the saved figure.
        highlight_idx: Index of method to highlight (e.g., RumorDiffusion).
        colors: List of colors for bars. If None, uses default colors.
    """
    fig, ax = plt.subplots()

    idx = np.arange(len(methods))
    
    if colors is None:
        bar_colors = [ICML_COLORS[2] if i != highlight_idx else ICML_COLORS[3] 
                     for i in range(len(methods))]
    else:
        bar_colors = colors
    
    bars = ax.bar(idx, means, yerr=cis, capsize=2.5, color=bar_colors)

    ax.set_ylabel("Macro-F1")
    ax.tick_params(top=False, right=False)

    ax.set_xticks(idx)
    ax.set_xticklabels(methods, rotation=45, ha="right")

    y_min = float(np.min(means - cis))
    y_max = float(np.max(means + cis))
    pad = max(0.001, 0.15 * (y_max - y_min))
    ax.set_ylim(y_min - pad, y_max + pad)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", format="pdf")
    plt.close(fig)


# -----------------------
# Generate bar charts for each transfer task
# -----------------------

# Twitter15→Twitter16 (use blue for Twitter15, same as hyperparameter sweep)
colors_t15_t16 = [ICML_COLORS[0] if i != 6 else ICML_COLORS[3] for i in range(7)]
save_transfer_bar_with_ci(
    methods=methods,
    means=t15_t16_mean,
    cis=t15_t16_ci,
    xlabel="",
    title=r"T15$\rightarrow$T16",
    out_path="Output/figures/transfer_t15_t16.pdf",
    highlight_idx=6,
    colors=colors_t15_t16,
)

# Twitter16→Twitter15 (use orange for Twitter16, same as hyperparameter sweep)
colors_t16_t15 = [ICML_COLORS[1] if i != 6 else ICML_COLORS[3] for i in range(7)]
save_transfer_bar_with_ci(
    methods=methods,
    means=t16_t15_mean,
    cis=t16_t15_ci,
    xlabel="",
    title=r"T16$\rightarrow$T15",
    out_path="Output/figures/transfer_t16_t15.pdf",
    highlight_idx=6,
    colors=colors_t16_t15,
)

# Weibo→PHEME
colors_weibo_pheme = [ICML_COLORS[2] if i != 6 else ICML_COLORS[3] for i in range(7)]
save_transfer_bar_with_ci(
    methods=methods,
    means=weibo_pheme_mean,
    cis=weibo_pheme_ci,
    xlabel="",
    title=r"Weibo$\rightarrow$PHEME",
    out_path="Output/figures/transfer_weibo_pheme.pdf",
    highlight_idx=6,
    colors=colors_weibo_pheme,
)

# PHEME→Weibo
colors_pheme_weibo = [ICML_COLORS[2] if i != 6 else ICML_COLORS[3] for i in range(7)]
save_transfer_bar_with_ci(
    methods=methods,
    means=pheme_weibo_mean,
    cis=pheme_weibo_ci,
    xlabel="",
    title=r"PHEME$\rightarrow$Weibo",
    out_path="Output/figures/transfer_pheme_weibo.pdf",
    highlight_idx=6,
    colors=colors_pheme_weibo,
)

print("Saved: transfer_t15_t16.pdf, transfer_t16_t15.pdf, transfer_weibo_pheme.pdf, transfer_pheme_weibo.pdf")



Saved: transfer_t15_t16.pdf, transfer_t16_t15.pdf, transfer_weibo_pheme.pdf, transfer_pheme_weibo.pdf
